# Chasing Ghosts in Ironman Data

This V2 notebook supports the flagship article *Chasing Ghosts in Ironman Data*.

The central change from the first version is that not every hard-limit flag is treated as an individual anomaly. A whole race can look "impossible" when a swim is shortened, a course is changed, or a timing convention differs from the nominal Ironman/70.3 format. This notebook therefore separates:

- `event_context_flag`: likely shortened/cancelled/altered leg or course-level timing convention
- `record_integrity_flag`: row arithmetic, duplicate time fields, transition outliers, or rank/time mismatch
- `individual_profile_flag`: unusual but plausible athlete profile after event-context and record-integrity issues are removed

## Table of Contents

1. [Setup](#setup)
2. [Load data and parse times](#load-data)
3. [Nominal distances, speeds, and event context](#event-context)
4. [Analysis table and relative percentiles](#features)
5. [Record integrity checks](#record-integrity)
6. [Hard limits after event-context adjustment](#hard-limits)
7. [Consistency profile checks](#consistency)
8. [Robust residual models](#model-flags)
9. [Combined review families](#combined)
10. [Article-facing tables and figures](#article-outputs)
11. [Case explanations](#cases)
12. [Validation](#validation)
13. [Article summary](#summary)

## 1. Setup <a id="setup"></a>

The notebook remains self-contained and reads the raw CSV files from `~/coachcox_results_csv`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
DATA_DIR = Path.home() / "coachcox_results_csv"

TIME_COLS = [
    "Overall Time",
    "Swim Time",
    "Bike Time",
    "Run Time",
    "Transition 1 Time",
    "Transition 2 Time",
]

LEG_CONFIG = {
    "swim": {"time_col": "Swim Time (s)", "rank_col": "Gender Swim Rank"},
    "bike": {"time_col": "Bike Time (s)", "rank_col": "Gender Bike Rank"},
    "run": {"time_col": "Run Time (s)", "rank_col": "Gender Run Rank"},
}

REL_COLS = ["swim_rel", "bike_rel", "run_rel"]
SPLIT_TIME_COLS = [cfg["time_col"] for cfg in LEG_CONFIG.values()]

DISTANCE_KM = {
    "70.3": {"swim": 1.9, "bike": 90.0, "run": 21.1},
    "full-distance": {"swim": 3.8, "bike": 180.0, "run": 42.2},
}

HARD_LIMITS = {
    "full-distance": {
        "swim_kmh": 6.2,
        "bike_kmh": 47.0,
        "run_kmh": 17.5,
        "overall_min_sec": 7 * 3600 + 20 * 60,
    },
    "70.3": {
        "swim_kmh": 7.0,
        "bike_kmh": 50.0,
        "run_kmh": 20.0,
        "overall_min_sec": 3 * 3600 + 20 * 60,
    },
}

MIN_PUBLIC_COHORT_SIZE = 150
RECONCILIATION_LIMIT_SEC = 5 * 60
DUPLICATE_TIME_TOLERANCE_SEC = 1
TRANSITION_REVIEW_SEC = 20 * 60
TRANSITION_EXTREME_SEC = 45 * 60

MISSING_OR_ZERO_RATE_THRESHOLD = 0.80
PROBABLE_CONTEXT_HARD_RATE = 0.80
POSSIBLE_CONTEXT_HARD_RATE = 0.30
PROBABLE_SHORT_RATIO = 0.55
POSSIBLE_SHORT_RATIO = 0.75
UNUSUALLY_LONG_RATIO = 1.35
WITHIN_EVENT_SPEED_Z_THRESHOLD = 4.0

BEST_REL_THRESHOLD = 0.95
SPREAD_REL_THRESHOLD = 0.70
MODEL_Z_THRESHOLD = 3.5
MIN_MODEL_ROWS = 500
MAX_MODEL_TRAIN_ROWS = 40_000
MAX_MODEL_CV_ROWS = 30_000
CV_SPLITS = 3

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"DATA_DIR: {DATA_DIR}")

## 2. Load data and parse times <a id="load-data"></a>

Event metadata is inferred from the file name. Time parsing keeps a raw seconds column for missing/zero diagnostics, then treats zero and negative durations as missing for analysis.

In [ ]:
def infer_event_metadata(path: Path) -> dict:
    base = path.stem
    race_year = base.split("__")[0]

    try:
        year = int(race_year[-4:])
        race = race_year[:-4]
    except ValueError:
        year = np.nan
        race = race_year

    stem_lower = base.lower()
    distance = "70.3" if ("70.3" in stem_lower or "70_3" in stem_lower) else "full-distance"

    return {
        "race": race,
        "year": year,
        "event_id": base,
        "distance": distance,
        "source_file": path.name,
    }


def load_results(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for key, value in infer_event_metadata(path).items():
        df[key] = value
    return df


def parse_time_columns(df: pd.DataFrame, time_cols: list[str]) -> pd.DataFrame:
    df = df.copy()

    for col in time_cols:
        raw_col = f"{col} raw seconds"
        sec_col = f"{col} (s)"
        raw_seconds = pd.to_timedelta(df[col], errors="coerce").dt.total_seconds()
        df[raw_col] = raw_seconds
        df[sec_col] = raw_seconds
        df.loc[df[sec_col] <= 0, sec_col] = np.nan

    return df


files = sorted(DATA_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}")

results = pd.concat((load_results(path) for path in files), ignore_index=True)
finishers = results.loc[results["Finish"].eq("FIN")].copy()
finishers = parse_time_columns(finishers, TIME_COLS)

print(f"Loaded files: {len(files):,}")
print(f"Raw rows: {len(results):,}")
print(f"Finisher rows: {len(finishers):,}")
finishers.head()

## 3. Nominal distances, speeds, and event context <a id="event-context"></a>

This section intentionally runs before athlete-level filtering. It asks whether a whole event-leg behaves unlike the nominal distance. If many athletes trigger the same hard swim flag, that is more likely a shortened swim than hundreds of individual anomalies.

In [ ]:
def robust_center_scale(values: pd.Series) -> tuple[float, float]:
    values = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        return np.nan, np.nan

    center = float(values.median())
    scale = float(1.4826 * np.median(np.abs(values - center)))

    if not np.isfinite(scale) or scale <= 1e-9:
        fallback = float(values.std())
        scale = fallback if np.isfinite(fallback) and fallback > 1e-9 else np.nan

    return center, scale


def add_nominal_distance_speed_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for leg, cfg in LEG_CONFIG.items():
        df[f"{leg}_km"] = df["distance"].map(
            {distance: distances[leg] for distance, distances in DISTANCE_KM.items()}
        )
        df[f"{leg}_kmh"] = df[f"{leg}_km"] / (df[cfg["time_col"]] / 3600.0)
        df.loc[~np.isfinite(df[f"{leg}_kmh"]), f"{leg}_kmh"] = np.nan
        df[f"{leg}_limit_kmh"] = df["distance"].map(
            {distance: limits[f"{leg}_kmh"] for distance, limits in HARD_LIMITS.items()}
        )
        df[f"{leg}_nominal_hard_flag"] = df[f"{leg}_kmh"].gt(df[f"{leg}_limit_kmh"]).fillna(False)
        raw_col = cfg["time_col"].replace(" (s)", " raw seconds")
        df[f"{leg}_missing_or_zero"] = df[raw_col].isna() | df[raw_col].le(0)

    df["overall_limit_sec"] = df["distance"].map(
        {distance: limits["overall_min_sec"] for distance, limits in HARD_LIMITS.items()}
    )
    df["overall_nominal_hard_flag"] = df["Overall Time (s)"].lt(df["overall_limit_sec"]).fillna(False)
    df["nominal_distance_hard_flag"] = df[
        ["swim_nominal_hard_flag", "bike_nominal_hard_flag", "run_nominal_hard_flag", "overall_nominal_hard_flag"]
    ].any(axis=1)

    return df


finishers = add_nominal_distance_speed_features(finishers)
finishers["complete_sbr"] = finishers[SPLIT_TIME_COLS].notna().all(axis=1)

cleaning_summary = pd.DataFrame(
    [
        {"stage": "Raw rows", "rows": len(results), "share_of_raw": 1.0},
        {"stage": "Finishers", "rows": len(finishers), "share_of_raw": len(finishers) / len(results)},
        {
            "stage": "Finishers with complete swim-bike-run",
            "rows": int(finishers["complete_sbr"].sum()),
            "share_of_raw": finishers["complete_sbr"].sum() / len(results),
        },
    ]
)

display(cleaning_summary)

In [ ]:
def build_duration_baselines(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for leg, cfg in LEG_CONFIG.items():
        time_col = cfg["time_col"]
        hard_col = f"{leg}_nominal_hard_flag"

        base = df.loc[df[time_col].notna() & ~df[hard_col]].copy()
        by_gender = (
            base.groupby(["distance", "gender"])[time_col]
            .median()
            .rename("baseline_median_s")
            .reset_index()
        )
        by_distance = (
            base.groupby("distance")[time_col]
            .median()
            .rename("distance_baseline_median_s")
            .reset_index()
        )

        merged = by_gender.merge(by_distance, on="distance", how="left")
        merged["leg"] = leg
        rows.append(merged)

    return pd.concat(rows, ignore_index=True)


duration_baselines = build_duration_baselines(finishers)
duration_baselines.head()

In [ ]:
def add_leg_duration_ratios(df: pd.DataFrame, baselines: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for leg, cfg in LEG_CONFIG.items():
        leg_base = baselines.loc[baselines["leg"].eq(leg)].drop(columns="leg")
        leg_base = leg_base.rename(
            columns={
                "baseline_median_s": f"{leg}_baseline_median_s",
                "distance_baseline_median_s": f"{leg}_distance_baseline_median_s",
            }
        )

        df = df.merge(leg_base, on=["distance", "gender"], how="left")
        baseline_col = f"{leg}_baseline_median_s"
        fallback_col = f"{leg}_distance_baseline_median_s"
        df[baseline_col] = df[baseline_col].fillna(df[fallback_col])
        df[f"{leg}_duration_ratio"] = df[cfg["time_col"]] / df[baseline_col]

    return df


def classify_event_leg(row: pd.Series) -> str:
    if row["missing_or_zero_rate"] >= MISSING_OR_ZERO_RATE_THRESHOLD:
        return "probable_cancelled_or_not_recorded"
    if row["nominal_hard_rate"] >= PROBABLE_CONTEXT_HARD_RATE or row["median_duration_ratio"] <= PROBABLE_SHORT_RATIO:
        return "probable_shortened_or_altered"
    if row["nominal_hard_rate"] >= POSSIBLE_CONTEXT_HARD_RATE or row["median_duration_ratio"] <= POSSIBLE_SHORT_RATIO:
        return "possible_shortened_or_altered"
    if row["median_duration_ratio"] >= UNUSUALLY_LONG_RATIO:
        return "unusually_slow_or_long"
    return "standard_like"


def build_event_leg_context(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for leg, cfg in LEG_CONFIG.items():
        time_col = cfg["time_col"]
        part = df[["event_id", "race", "year", "distance", "gender", time_col, f"{leg}_missing_or_zero", f"{leg}_nominal_hard_flag", f"{leg}_duration_ratio"]].copy()
        part["leg"] = leg
        part = part.rename(
            columns={
                time_col: "duration_s",
                f"{leg}_missing_or_zero": "missing_or_zero",
                f"{leg}_nominal_hard_flag": "nominal_hard_flag",
                f"{leg}_duration_ratio": "duration_ratio",
            }
        )
        rows.append(part)

    long = pd.concat(rows, ignore_index=True)

    context = (
        long.groupby(["event_id", "race", "year", "distance", "leg"], dropna=False)
        .agg(
            rows=("duration_s", "size"),
            valid_rows=("duration_s", "count"),
            missing_or_zero_rate=("missing_or_zero", "mean"),
            nominal_hard_rate=("nominal_hard_flag", "mean"),
            median_duration_s=("duration_s", "median"),
            median_duration_ratio=("duration_ratio", "median"),
        )
        .reset_index()
    )

    context["event_leg_context_category"] = context.apply(classify_event_leg, axis=1)
    context["event_leg_context_flag"] = context["event_leg_context_category"].ne("standard_like")

    return context


finishers = add_leg_duration_ratios(finishers, duration_baselines)
event_leg_context = build_event_leg_context(finishers)

event_context_candidates = (
    event_leg_context.loc[event_leg_context["event_leg_context_flag"]]
    .sort_values(["event_leg_context_category", "nominal_hard_rate", "missing_or_zero_rate"], ascending=[True, False, False])
    .reset_index(drop=True)
)

event_context_candidates.head(20)

## 4. Analysis table and relative percentiles <a id="features"></a>

Athlete-level checks are run only on finishers with complete swim-bike-run times. Event context is merged back in as a feature rather than discarded.

In [ ]:
def add_event_context_columns(df: pd.DataFrame, context: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    affected_parts = []
    for leg in LEG_CONFIG:
        leg_context = context.loc[context["leg"].eq(leg), ["event_id", "event_leg_context_category", "event_leg_context_flag", "nominal_hard_rate", "median_duration_ratio"]].copy()
        leg_context = leg_context.rename(
            columns={
                "event_leg_context_category": f"{leg}_event_context_category",
                "event_leg_context_flag": f"{leg}_event_context_flag",
                "nominal_hard_rate": f"{leg}_event_nominal_hard_rate",
                "median_duration_ratio": f"{leg}_event_median_duration_ratio",
            }
        )
        df = df.merge(leg_context, on="event_id", how="left")
        df[f"{leg}_event_context_flag"] = df[f"{leg}_event_context_flag"].fillna(False)
        affected_parts.append(f"{leg}_event_context_flag")

    df["event_context_flag"] = df[affected_parts].any(axis=1)
    df["event_context_affected_legs"] = df.apply(
        lambda row: ", ".join([leg for leg in LEG_CONFIG if bool(row[f"{leg}_event_context_flag"])]) or "none",
        axis=1,
    )

    return df


def add_relative_strength(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    group_cols = ["event_id", "gender"]

    for time_col, rel_col in [
        ("Overall Time (s)", "overall_rel"),
        ("Swim Time (s)", "swim_rel"),
        ("Bike Time (s)", "bike_rel"),
        ("Run Time (s)", "run_rel"),
    ]:
        pct = df.groupby(group_cols)[time_col].rank(method="average", pct=True, ascending=True)
        df[rel_col] = 1 - pct

    return df


analysis_df = finishers.loc[finishers["complete_sbr"]].copy()
analysis_df = add_event_context_columns(analysis_df, event_leg_context)
analysis_df = add_relative_strength(analysis_df)

event_gender_counts = (
    analysis_df.groupby(["event_id", "gender"])
    .size()
    .rename("cohort_valid_sbr_finishers")
    .reset_index()
)
analysis_df = analysis_df.merge(event_gender_counts, on=["event_id", "gender"], how="left")

print(f"Analysis rows with complete SBR: {len(analysis_df):,}")
analysis_df[["race", "year", "distance", "gender", "event_context_flag", "event_context_affected_legs", "swim_rel", "bike_rel", "run_rel"]].head()

## 5. Record integrity checks <a id="record-integrity"></a>

These flags look for rows where the arithmetic or source fields do not describe a coherent result.

In [ ]:
def add_record_integrity_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    split_cols = ["Swim Time (s)", "Bike Time (s)", "Run Time (s)"]
    transition_cols = ["Transition 1 Time (s)", "Transition 2 Time (s)"]

    df["split_sum_no_transition_s"] = df[split_cols].sum(axis=1, min_count=len(split_cols))
    df["split_sum_with_transition_s"] = df[split_cols + transition_cols].sum(axis=1, min_count=len(split_cols) + len(transition_cols))
    df["overall_delta_without_transitions_s"] = df["Overall Time (s)"] - df["split_sum_no_transition_s"]
    df["overall_delta_with_transitions_s"] = df["Overall Time (s)"] - df["split_sum_with_transition_s"]
    df["overall_abs_delta_with_transitions_s"] = df["overall_delta_with_transitions_s"].abs()
    df["reconciliation_flag"] = df["overall_abs_delta_with_transitions_s"].gt(RECONCILIATION_LIMIT_SEC).fillna(False)

    duplicate_parts = []
    for leg, cfg in LEG_CONFIG.items():
        col = f"{leg}_equals_overall_flag"
        df[col] = (df[cfg["time_col"]] - df["Overall Time (s)"]).abs().le(DUPLICATE_TIME_TOLERANCE_SEC).fillna(False)
        duplicate_parts.append(col)
    df["duplicate_time_flag"] = df[duplicate_parts].any(axis=1)

    df["t1_review_flag"] = df["Transition 1 Time (s)"].gt(TRANSITION_REVIEW_SEC).fillna(False)
    df["t2_review_flag"] = df["Transition 2 Time (s)"].gt(TRANSITION_REVIEW_SEC).fillna(False)
    df["transition_review_flag"] = df["t1_review_flag"] | df["t2_review_flag"]
    df["t1_extreme_flag"] = df["Transition 1 Time (s)"].gt(TRANSITION_EXTREME_SEC).fillna(False)
    df["t2_extreme_flag"] = df["Transition 2 Time (s)"].gt(TRANSITION_EXTREME_SEC).fillna(False)
    df["transition_extreme_flag"] = df["t1_extreme_flag"] | df["t2_extreme_flag"]

    rank_specs = [
        ("overall", "Overall Time (s)", "Gender Rank"),
        ("swim", "Swim Time (s)", "Gender Swim Rank"),
        ("bike", "Bike Time (s)", "Gender Bike Rank"),
        ("run", "Run Time (s)", "Gender Run Rank"),
    ]

    mismatch_cols = []
    for prefix, time_col, rank_col in rank_specs:
        numeric_rank_col = f"{prefix}_source_gender_rank"
        derived_rank_col = f"{prefix}_derived_gender_rank"
        mismatch_col = f"{prefix}_rank_time_mismatch_flag"

        df[numeric_rank_col] = pd.to_numeric(df[rank_col], errors="coerce")
        df[derived_rank_col] = df.groupby(["event_id", "gender"])[time_col].rank(method="min", ascending=True)
        tolerance = np.maximum(10, df["cohort_valid_sbr_finishers"].fillna(0) * 0.02)
        df[mismatch_col] = (
            df[numeric_rank_col].notna()
            & df[derived_rank_col].notna()
            & (df[numeric_rank_col] - df[derived_rank_col]).abs().gt(tolerance)
        )
        mismatch_cols.append(mismatch_col)

    df["rank_time_mismatch_flag"] = df[mismatch_cols].any(axis=1)
    df["record_integrity_flag"] = df[
        ["reconciliation_flag", "duplicate_time_flag", "transition_review_flag", "transition_extreme_flag", "rank_time_mismatch_flag"]
    ].any(axis=1)

    return df


analysis_df = add_record_integrity_flags(analysis_df)

record_integrity_summary = (
    analysis_df[["reconciliation_flag", "duplicate_time_flag", "transition_review_flag", "transition_extreme_flag", "rank_time_mismatch_flag", "record_integrity_flag"]]
    .mean()
    .mul(100)
    .round(4)
)
record_integrity_summary

## 6. Hard limits after event-context adjustment <a id="hard-limits"></a>

Nominal hard flags answer: "Is this impossible if the leg was the standard distance?"

Individual hard flags answer: "Is this still extreme after accounting for likely course-level context?"

In [ ]:
def add_within_event_speed_z(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for leg in LEG_CONFIG:
        speed_col = f"{leg}_kmh"
        z_col = f"{leg}_within_event_speed_robust_z"

        stats = (
            df.groupby(["event_id", "gender"])[speed_col]
            .agg(lambda s: robust_center_scale(s)[0])
            .rename("center")
            .reset_index()
        )
        scales = (
            df.groupby(["event_id", "gender"])[speed_col]
            .agg(lambda s: robust_center_scale(s)[1])
            .rename("scale")
            .reset_index()
        )
        stats = stats.merge(scales, on=["event_id", "gender"], how="left")
        stats = stats.rename(columns={"center": f"{leg}_speed_center", "scale": f"{leg}_speed_scale"})
        df = df.merge(stats, on=["event_id", "gender"], how="left")
        df[z_col] = (df[speed_col] - df[f"{leg}_speed_center"]) / df[f"{leg}_speed_scale"]

    return df


def add_adjusted_hard_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for leg in LEG_CONFIG:
        df[f"{leg}_individual_hard_flag"] = (
            df[f"{leg}_nominal_hard_flag"]
            & (
                ~df[f"{leg}_event_context_flag"]
                | df[f"{leg}_within_event_speed_robust_z"].gt(WITHIN_EVENT_SPEED_Z_THRESHOLD).fillna(False)
            )
        )

    df["overall_individual_hard_flag"] = df["overall_nominal_hard_flag"] & ~df["event_context_flag"]
    df["individual_hard_flag"] = df[
        ["swim_individual_hard_flag", "bike_individual_hard_flag", "run_individual_hard_flag", "overall_individual_hard_flag"]
    ].any(axis=1)

    return df


analysis_df = add_within_event_speed_z(analysis_df)
analysis_df = add_adjusted_hard_flags(analysis_df)

before_after_hard_summary = pd.DataFrame(
    [
        {"flag_family": "Nominal hard flags", "rows": int(analysis_df["nominal_distance_hard_flag"].sum()), "rate": analysis_df["nominal_distance_hard_flag"].mean()},
        {"flag_family": "Individual hard flags after context adjustment", "rows": int(analysis_df["individual_hard_flag"].sum()), "rate": analysis_df["individual_hard_flag"].mean()},
    ]
).assign(rate_pct=lambda d: d["rate"] * 100)

before_after_hard_summary

## 7. Consistency profile checks <a id="consistency"></a>

Consistency is now split into profile types and marked low-confidence when a likely altered leg is involved.

In [ ]:
def add_consistency_profile_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["spread_rel"] = df[REL_COLS].max(axis=1) - df[REL_COLS].min(axis=1)
    df["best_rel"] = df[REL_COLS].max(axis=1)
    df["worst_rel"] = df[REL_COLS].min(axis=1)
    df["best_leg"] = df[REL_COLS].idxmax(axis=1).str.replace("_rel", "", regex=False)
    df["worst_leg"] = df[REL_COLS].idxmin(axis=1).str.replace("_rel", "", regex=False)

    reliable_values = []
    reliable_counts = []
    reliable_best = []
    reliable_worst = []
    reliable_second_best = []
    reliable_spread = []

    for _, row in df.iterrows():
        vals = []
        for leg in LEG_CONFIG:
            if not bool(row[f"{leg}_event_context_flag"]):
                val = row[f"{leg}_rel"]
                if pd.notna(val):
                    vals.append(float(val))

        reliable_counts.append(len(vals))
        if vals:
            sorted_vals = sorted(vals, reverse=True)
            reliable_best.append(sorted_vals[0])
            reliable_worst.append(sorted_vals[-1])
            reliable_second_best.append(sorted_vals[1] if len(sorted_vals) > 1 else np.nan)
            reliable_spread.append(sorted_vals[0] - sorted_vals[-1])
        else:
            reliable_best.append(np.nan)
            reliable_worst.append(np.nan)
            reliable_second_best.append(np.nan)
            reliable_spread.append(np.nan)

    df["reliable_leg_count"] = reliable_counts
    df["reliable_best_rel"] = reliable_best
    df["reliable_worst_rel"] = reliable_worst
    df["reliable_second_best_rel"] = reliable_second_best
    df["reliable_spread_rel"] = reliable_spread

    df["raw_consistency_flag"] = df["best_rel"].ge(BEST_REL_THRESHOLD) & df["spread_rel"].ge(SPREAD_REL_THRESHOLD)
    df["profile_context_low_confidence"] = df["event_context_flag"]
    df["consistency_flag"] = (
        df["reliable_leg_count"].ge(3)
        & df["reliable_best_rel"].ge(BEST_REL_THRESHOLD)
        & df["reliable_spread_rel"].ge(SPREAD_REL_THRESHOLD)
    )

    df["one_leg_spike_flag"] = df["consistency_flag"] & df["reliable_second_best_rel"].le(0.65).fillna(False)
    df["one_leg_collapse_flag"] = (
        df["reliable_leg_count"].ge(3)
        & df["reliable_worst_rel"].le(0.05)
        & df["reliable_best_rel"].ge(0.65)
        & df["reliable_spread_rel"].ge(SPREAD_REL_THRESHOLD)
    )
    df["multi_leg_gap_flag"] = df["consistency_flag"] & ~df["one_leg_spike_flag"] & ~df["one_leg_collapse_flag"]
    df["consistency_confidence"] = np.where(df["profile_context_low_confidence"], "low", "medium")

    return df


analysis_df = add_consistency_profile_flags(analysis_df)

consistency_profile_summary = (
    analysis_df[["raw_consistency_flag", "consistency_flag", "one_leg_spike_flag", "one_leg_collapse_flag", "multi_leg_gap_flag"]]
    .mean()
    .mul(100)
    .round(4)
)
consistency_profile_summary

## 8. Robust residual models <a id="model-flags"></a>

Huber models are trained only on rows without event-context or record-integrity problems. The model remains deliberately interpretable: one leg is predicted from the other two plus division, never from `overall_rel`.

In [ ]:
def validation_metrics_grouped(x: pd.DataFrame, y: pd.Series, groups: pd.Series) -> tuple[float, float]:
    if len(x) < MIN_MODEL_ROWS or groups.nunique() < 2:
        return np.nan, np.nan

    if len(x) > MAX_MODEL_CV_ROWS:
        sample_idx = x.sample(MAX_MODEL_CV_ROWS, random_state=RANDOM_STATE).index
        x = x.loc[sample_idx]
        y = y.loc[sample_idx]
        groups = groups.loc[sample_idx]

    n_splits = min(CV_SPLITS, groups.nunique())
    splitter = GroupKFold(n_splits=n_splits)

    maes = []
    r2s = []
    for train_idx, test_idx in splitter.split(x, y, groups):
        if len(train_idx) < MIN_MODEL_ROWS // 2 or len(test_idx) == 0:
            continue
        model = HuberRegressor(max_iter=100)
        model.fit(x.iloc[train_idx].to_numpy(), y.iloc[train_idx].to_numpy())
        pred = model.predict(x.iloc[test_idx].to_numpy())
        actual = y.iloc[test_idx].to_numpy()
        maes.append(mean_absolute_error(actual, pred))
        if len(np.unique(actual)) > 1:
            r2s.append(r2_score(actual, pred))

    return (float(np.mean(maes)) if maes else np.nan, float(np.mean(r2s)) if r2s else np.nan)


def model_confidence(n_train: int, val_mae: float, resid_scale: float) -> str:
    if n_train >= 5_000 and np.isfinite(val_mae) and val_mae <= 0.12 and np.isfinite(resid_scale) and resid_scale <= 0.10:
        return "high"
    if n_train >= 1_000 and (not np.isfinite(val_mae) or val_mae <= 0.18):
        return "medium"
    return "low"


def fit_leg_residual_models(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    model_summaries = []

    for leg in LEG_CONFIG:
        df[f"{leg}_model_pred"] = np.nan
        df[f"{leg}_model_resid"] = np.nan
        df[f"{leg}_model_robust_z"] = np.nan
        df[f"{leg}_model_flag"] = False
        df[f"{leg}_model_confidence"] = "not_fit"

    trainable_base = ~df["event_context_flag"] & ~df["record_integrity_flag"]

    for distance in sorted(df["distance"].dropna().unique()):
        for gender in sorted(df.loc[df["distance"].eq(distance), "gender"].dropna().unique()):
            group_mask = df["distance"].eq(distance) & df["gender"].eq(gender)

            for target_leg in ["swim", "bike", "run"]:
                target_col = f"{target_leg}_rel"
                predictor_cols = [f"{leg}_rel" for leg in LEG_CONFIG if leg != target_leg]
                base_cols = predictor_cols + [target_col, "Division", "event_id"]

                score_mask = group_mask & df[base_cols].notna().all(axis=1)
                train_mask = score_mask & trainable_base

                score_idx = df.index[score_mask]
                train_idx = df.index[train_mask]

                if len(score_idx) < MIN_MODEL_ROWS or len(train_idx) < MIN_MODEL_ROWS:
                    model_summaries.append(
                        {
                            "distance": distance,
                            "gender": gender,
                            "target_leg": target_leg,
                            "status": "skipped_too_few_clean_rows",
                            "n_scored": len(score_idx),
                            "n_train": len(train_idx),
                            "n_flagged": 0,
                            "flag_rate": np.nan,
                            "resid_scale": np.nan,
                            "mae_train": np.nan,
                            "r2_train": np.nan,
                            "group_cv_mae": np.nan,
                            "group_cv_r2": np.nan,
                            "model_confidence": "not_fit",
                        }
                    )
                    continue

                score_frame = df.loc[score_idx, base_cols].copy()
                train_frame = df.loc[train_idx, base_cols].copy()

                score_frame["Division"] = score_frame["Division"].fillna("Unknown").astype(str)
                train_frame["Division"] = train_frame["Division"].fillna("Unknown").astype(str)

                if len(train_frame) > MAX_MODEL_TRAIN_ROWS:
                    train_frame = train_frame.sample(MAX_MODEL_TRAIN_ROWS, random_state=RANDOM_STATE)

                x_score = pd.get_dummies(score_frame[predictor_cols + ["Division"]], columns=["Division"], drop_first=True, dtype=float)
                x_train = pd.get_dummies(train_frame[predictor_cols + ["Division"]], columns=["Division"], drop_first=True, dtype=float)
                x_score, x_train = x_score.align(x_train, join="outer", axis=1, fill_value=0.0)
                x_train = x_train[x_score.columns]

                y_train = train_frame[target_col].astype(float)

                val_mae, val_r2 = validation_metrics_grouped(x_train, y_train, train_frame["event_id"])

                model = HuberRegressor(max_iter=150)
                model.fit(x_train.to_numpy(), y_train.to_numpy())

                train_pred = model.predict(x_train.to_numpy())
                train_resid = y_train.to_numpy() - train_pred
                resid_center, resid_scale = robust_center_scale(pd.Series(train_resid))

                pred = model.predict(x_score.to_numpy())
                actual = score_frame[target_col].astype(float).to_numpy()
                resid = actual - pred

                if np.isfinite(resid_scale):
                    robust_z = (resid - resid_center) / resid_scale
                    flags = np.abs(robust_z) > MODEL_Z_THRESHOLD
                else:
                    robust_z = np.full_like(resid, np.nan, dtype=float)
                    flags = np.zeros_like(resid, dtype=bool)

                confidence = model_confidence(len(train_frame), val_mae, resid_scale)

                df.loc[score_idx, f"{target_leg}_model_pred"] = pred
                df.loc[score_idx, f"{target_leg}_model_resid"] = resid
                df.loc[score_idx, f"{target_leg}_model_robust_z"] = robust_z
                df.loc[score_idx, f"{target_leg}_model_flag"] = flags
                df.loc[score_idx, f"{target_leg}_model_confidence"] = confidence

                model_summaries.append(
                    {
                        "distance": distance,
                        "gender": gender,
                        "target_leg": target_leg,
                        "status": "fit",
                        "n_scored": len(score_idx),
                        "n_train": len(train_frame),
                        "n_flagged": int(flags.sum()),
                        "flag_rate": float(flags.mean()),
                        "resid_scale": float(resid_scale) if np.isfinite(resid_scale) else np.nan,
                        "mae_train": float(mean_absolute_error(y_train, train_pred)),
                        "r2_train": float(r2_score(y_train, train_pred)),
                        "group_cv_mae": val_mae,
                        "group_cv_r2": val_r2,
                        "model_confidence": confidence,
                    }
                )

    return df, pd.DataFrame(model_summaries)


analysis_df, model_summary = fit_leg_residual_models(analysis_df)
model_summary

## 9. Combined review families <a id="combined"></a>

The final flags are grouped by interpretation, not by drama. This is the main article improvement.

In [ ]:
MODEL_FLAG_COLS = ["swim_model_flag", "bike_model_flag", "run_model_flag"]
PROFILE_FLAG_COLS = ["individual_hard_flag", "consistency_flag"] + MODEL_FLAG_COLS

analysis_df["model_layer_flag"] = analysis_df[MODEL_FLAG_COLS].any(axis=1)
analysis_df["individual_profile_flag"] = (
    ~analysis_df["event_context_flag"]
    & ~analysis_df["record_integrity_flag"]
    & analysis_df[PROFILE_FLAG_COLS].any(axis=1)
)

analysis_df["any_review_flag_v2"] = analysis_df[
    ["event_context_flag", "record_integrity_flag", "individual_profile_flag"]
].any(axis=1)
analysis_df["review_family_count"] = analysis_df[
    ["event_context_flag", "record_integrity_flag", "individual_profile_flag"]
].sum(axis=1).astype(int)
analysis_df["max_model_abs_z"] = analysis_df[
    ["swim_model_robust_z", "bike_model_robust_z", "run_model_robust_z"]
].abs().max(axis=1)

review_family_summary = (
    analysis_df[["event_context_flag", "record_integrity_flag", "individual_profile_flag", "any_review_flag_v2"]]
    .mean()
    .mul(100)
    .round(4)
)

review_family_summary

In [ ]:
flag_taxonomy = pd.DataFrame(
    [
        {
            "flag_family": "Event/course context",
            "main_columns": "event_context_flag, *_event_context_category",
            "plain_language_question": "Does a whole race-leg behave as if the course was shortened, cancelled, unusually long, or recorded differently?",
            "article_interpretation": "This is mostly a race-context signal, not an individual-athlete signal.",
            "review_action": "Treat nominal-distance speed and overall-time comparisons with caution for that event-leg.",
        },
        {
            "flag_family": "Record integrity",
            "main_columns": "reconciliation_flag, duplicate_time_flag, transition_*_flag, rank_time_mismatch_flag",
            "plain_language_question": "Does the row describe a coherent race result?",
            "article_interpretation": "This is the strongest data-quality layer because the row contradicts itself.",
            "review_action": "Check the raw result before using the row in modelling or examples.",
        },
        {
            "flag_family": "Individual hard limit",
            "main_columns": "individual_hard_flag",
            "plain_language_question": "Is a row still implausibly fast after course-context flags are considered?",
            "article_interpretation": "This is a higher-confidence individual review signal than the nominal hard flag.",
            "review_action": "Inspect alongside event context and source timing.",
        },
        {
            "flag_family": "Consistency profile",
            "main_columns": "consistency_flag, one_leg_spike_flag, one_leg_collapse_flag, multi_leg_gap_flag",
            "plain_language_question": "Is one reliable leg extremely different from the other reliable legs?",
            "article_interpretation": "This can reflect real specialisation, fatigue, pacing, injury, or a split issue.",
            "review_action": "Use as a profile-pattern signal rather than a verdict.",
        },
        {
            "flag_family": "Robust residual model",
            "main_columns": "*_model_flag, *_model_robust_z, *_model_confidence",
            "plain_language_question": "Is a leg much different from what the other two legs and division suggest?",
            "article_interpretation": "This is the highest-uncertainty layer, useful mostly when it overlaps with other evidence.",
            "review_action": "Explain cautiously and include model confidence.",
        },
    ]
)

flag_taxonomy

## 10. Article-facing tables and figures <a id="article-outputs"></a>

These outputs are designed to make the improved story visible: some anomalies are rows, some are athletes, and some are whole races.

In [ ]:
course_context_candidates = (
    event_context_candidates.loc[
        event_context_candidates["rows"].ge(MIN_PUBLIC_COHORT_SIZE),
        [
            "race",
            "year",
            "distance",
            "leg",
            "rows",
            "valid_rows",
            "event_leg_context_category",
            "missing_or_zero_rate",
            "nominal_hard_rate",
            "median_duration_ratio",
        ],
    ]
    .sort_values(["event_leg_context_category", "nominal_hard_rate", "missing_or_zero_rate"], ascending=[True, False, False])
    .reset_index(drop=True)
)

individual_review_summary = (
    analysis_df.groupby(["distance", "gender"], dropna=False)
    .agg(
        n=("event_id", "size"),
        event_context_rate=("event_context_flag", "mean"),
        record_integrity_rate=("record_integrity_flag", "mean"),
        nominal_hard_rate=("nominal_distance_hard_flag", "mean"),
        individual_hard_rate=("individual_hard_flag", "mean"),
        consistency_rate=("consistency_flag", "mean"),
        model_rate=("model_layer_flag", "mean"),
        individual_profile_rate=("individual_profile_flag", "mean"),
        any_review_rate_v2=("any_review_flag_v2", "mean"),
    )
    .reset_index()
)

race_summary_v2 = (
    analysis_df.groupby(["event_id", "race", "year", "distance", "gender"], dropna=False)
    .agg(
        n=("event_id", "size"),
        event_context_rate=("event_context_flag", "mean"),
        record_integrity_rate=("record_integrity_flag", "mean"),
        individual_profile_rate=("individual_profile_flag", "mean"),
        any_review_rate_v2=("any_review_flag_v2", "mean"),
        nominal_hard_rate=("nominal_distance_hard_flag", "mean"),
        individual_hard_rate=("individual_hard_flag", "mean"),
    )
    .reset_index()
)

public_race_summary_v2 = race_summary_v2.loc[race_summary_v2["n"].ge(MIN_PUBLIC_COHORT_SIZE)].copy()
public_race_summary_v2["individual_profile_rate_pct"] = public_race_summary_v2["individual_profile_rate"] * 100

display(course_context_candidates.head(20).round(3))
display(individual_review_summary.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = before_after_hard_summary.copy()
ax.bar(plot_df["flag_family"], plot_df["rate_pct"])
ax.set_ylabel("Rows flagged (%)")
ax.set_title("Hard flags before and after event-context adjustment")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()

In [ ]:
top_context = course_context_candidates.head(15).copy()
if not top_context.empty:
    labels = (
        top_context["race"].astype(str)
        + " "
        + top_context["year"].astype("Int64").astype(str)
        + " / "
        + top_context["leg"].astype(str)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(labels[::-1], (top_context["nominal_hard_rate"] * 100)[::-1])
    ax.set_xlabel("Nominal hard rate (%)")
    ax.set_ylabel("")
    ax.set_title("Course-context candidates, not individual accusations")
    plt.tight_layout()
else:
    print("No course-context candidates above the public cohort threshold.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
vals = analysis_df.loc[~analysis_df["event_context_flag"], "reliable_spread_rel"].dropna()
ax.hist(vals, bins=70, alpha=0.75)
ax.axvline(SPREAD_REL_THRESHOLD, color="black", linestyle="--", linewidth=1.2)
ax.set_xlabel("Reliable-leg spread between best and worst split percentile")
ax.set_ylabel("Number of finishers")
ax.set_title("Consistency spread after removing likely course-context legs")
plt.tight_layout()

In [ ]:
layer_plot = individual_review_summary.copy()
layer_plot["event_context_pct"] = layer_plot["event_context_rate"] * 100
layer_plot["record_integrity_pct"] = layer_plot["record_integrity_rate"] * 100
layer_plot["individual_profile_pct"] = layer_plot["individual_profile_rate"] * 100

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(layer_plot))
width = 0.25
labels = layer_plot["distance"].astype(str) + " / " + layer_plot["gender"].astype(str)

ax.bar(x - width, layer_plot["event_context_pct"], width, label="event/course context")
ax.bar(x, layer_plot["record_integrity_pct"], width, label="record integrity")
ax.bar(x + width, layer_plot["individual_profile_pct"], width, label="individual profile")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20)
ax.set_ylabel("Rows flagged (%)")
ax.set_title("Review families after context adjustment")
ax.legend()
plt.tight_layout()

In [ ]:
def sensitivity_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for hard_threshold in [0.30, 0.50, 0.80]:
        context_flag = (
            df["swim_event_nominal_hard_rate"].ge(hard_threshold)
            | df["bike_event_nominal_hard_rate"].ge(hard_threshold)
            | df["run_event_nominal_hard_rate"].ge(hard_threshold)
        ).fillna(False)
        rows.append(
            {
                "threshold_family": "event_nominal_hard_rate",
                "threshold": hard_threshold,
                "flag_rate": context_flag.mean(),
                "flagged_rows": int(context_flag.sum()),
            }
        )

    for spread_threshold in [0.65, 0.70, 0.75, 0.80]:
        spread_flag = (
            ~df["event_context_flag"]
            & df["reliable_best_rel"].ge(BEST_REL_THRESHOLD)
            & df["reliable_spread_rel"].ge(spread_threshold)
        )
        rows.append(
            {
                "threshold_family": "consistency_spread",
                "threshold": spread_threshold,
                "flag_rate": spread_flag.mean(),
                "flagged_rows": int(spread_flag.sum()),
            }
        )

    for z_threshold in [3.0, 3.5, 4.0]:
        model_flag = (~df["event_context_flag"] & ~df["record_integrity_flag"] & df["max_model_abs_z"].gt(z_threshold).fillna(False))
        rows.append(
            {
                "threshold_family": "model_abs_robust_z",
                "threshold": z_threshold,
                "flag_rate": model_flag.mean(),
                "flagged_rows": int(model_flag.sum()),
            }
        )

    return pd.DataFrame(rows)


sensitivity = sensitivity_table(analysis_df)
sensitivity.assign(flag_rate_pct=lambda d: d["flag_rate"] * 100)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for family, group in sensitivity.groupby("threshold_family"):
    ax.plot(group["threshold"], group["flag_rate"] * 100, marker="o", label=family.replace("_", " "))
ax.set_xlabel("Threshold")
ax.set_ylabel("Rows flagged (%)")
ax.set_title("Sensitivity of review rates to threshold choices")
ax.legend()
plt.tight_layout()

## 11. Case explanations <a id="cases"></a>

These cases mirror the original example types, but the interpretation is now more careful. Race names, event IDs, names, bibs, and countries are excluded from the article-facing table.

In [ ]:
CASE_SELECTORS = [
    {"case_id": "Case A", "event_id": "ironmancopenhagen2017__388", "gender": "Male", "division": "M30-34", "overall_time": "11:52:26"},
    {"case_id": "Case B", "event_id": "ironman70.3boulder2013__1276", "gender": "Male", "division": "M25-29", "overall_time": "06:15:18"},
    {"case_id": "Case C", "event_id": "ironman70.3jönköping2024__2279", "gender": "Male", "division": "M35-39", "overall_time": "04:31:30"},
    {"case_id": "Case D", "event_id": "ironmancanada2018__439", "gender": "Female", "division": "F40-44", "overall_time": "15:41:49"},
    {"case_id": "Case E", "event_id": "ironman70.3neworleans2014__1339", "gender": "Male", "division": "M30-34", "overall_time": "05:28:09"},
]


def find_case_row(df: pd.DataFrame, selector: dict) -> pd.DataFrame:
    mask = (
        df["event_id"].eq(selector["event_id"])
        & df["gender"].eq(selector["gender"])
        & df["Division"].eq(selector["division"])
        & df["Overall Time"].eq(selector["overall_time"])
    )
    match = df.loc[mask].head(1).copy()
    if not match.empty:
        match["case_id"] = selector["case_id"]
    return match


case_frames = [find_case_row(analysis_df, selector) for selector in CASE_SELECTORS]
case_source = pd.concat([frame for frame in case_frames if not frame.empty], ignore_index=True)

fallback_specs = [
    ("Case A", analysis_df["individual_profile_flag"] & analysis_df["consistency_flag"] & analysis_df["model_layer_flag"]),
    ("Case B", analysis_df["record_integrity_flag"] & (analysis_df["duplicate_time_flag"] | analysis_df["transition_extreme_flag"])),
    ("Case C", analysis_df["event_context_flag"] & analysis_df["nominal_distance_hard_flag"] & ~analysis_df["individual_hard_flag"]),
    ("Case D", analysis_df["individual_profile_flag"] & analysis_df["consistency_flag"] & ~analysis_df["model_layer_flag"]),
    ("Case E", analysis_df["individual_profile_flag"] & analysis_df["model_layer_flag"] & ~analysis_df["consistency_flag"]),
]

existing_case_ids = set(case_source["case_id"]) if not case_source.empty else set()
for case_id, mask in fallback_specs:
    if case_id in existing_case_ids:
        continue
    candidates = analysis_df.loc[mask].copy()
    if not candidates.empty:
        sampled = candidates.sample(1, random_state=100 + ord(case_id[-1])).assign(case_id=case_id)
        case_source = pd.concat([case_source, sampled], ignore_index=True)

case_source = case_source.sort_values("case_id").reset_index(drop=True)
case_source[["case_id", "distance", "gender", "cohort_valid_sbr_finishers", "event_context_flag", "record_integrity_flag", "individual_profile_flag", "swim_rel", "bike_rel", "run_rel", "spread_rel", "max_model_abs_z"]]

In [ ]:
def confidence_for_case(row: pd.Series) -> str:
    if bool(row["record_integrity_flag"]):
        return "high"
    if bool(row["event_context_flag"]) and bool(row["nominal_distance_hard_flag"]) and not bool(row["individual_hard_flag"]):
        return "high"
    if bool(row["individual_profile_flag"]) and bool(row["consistency_flag"]) and bool(row["model_layer_flag"]):
        return "medium"
    if bool(row["individual_profile_flag"]):
        return "low-medium"
    return "low"


def explain_case(row: pd.Series) -> dict:
    affected = row["event_context_affected_legs"]
    model_legs = [leg for leg in LEG_CONFIG if bool(row[f"{leg}_model_flag"])]

    if bool(row["record_integrity_flag"]):
        reasons = []
        if bool(row["duplicate_time_flag"]):
            reasons.append("one split duplicates the overall time")
        if bool(row["transition_extreme_flag"]):
            reasons.append("a transition time is extremely long")
        if bool(row["reconciliation_flag"]):
            reasons.append("the sum of parts does not reconcile with the overall time")
        if bool(row["rank_time_mismatch_flag"]):
            reasons.append("rank fields disagree with sorted times")
        why = "; ".join(reasons) or "the row contradicts itself"
        return {
            "primary_interpretation": "Record integrity issue",
            "why_flagged": why,
            "caveat": "This is about the source row, not athlete intent.",
            "confidence": "high",
            "article_safe_sentence": "This case is best read as a timing-record problem because the row does not add up internally.",
        }

    if bool(row["event_context_flag"]) and bool(row["nominal_distance_hard_flag"]) and not bool(row["individual_hard_flag"]):
        return {
            "primary_interpretation": "Event/course context",
            "why_flagged": f"The nominal hard flag appears on a likely altered event leg ({affected}).",
            "caveat": "The athlete may simply have completed a shortened or otherwise altered course.",
            "confidence": "high",
            "article_safe_sentence": "This case shows why hard limits need event context: the row looks impossible only if we assume the standard distance.",
        }

    if bool(row["individual_profile_flag"]):
        layers = []
        if bool(row["individual_hard_flag"]):
            layers.append("adjusted hard limit")
        if bool(row["consistency_flag"]):
            layers.append("large reliable-leg spread")
        if bool(row["model_layer_flag"]):
            layers.append("robust model residual" + (f" on {', '.join(model_legs)}" if model_legs else ""))
        why = "; ".join(layers)
        return {
            "primary_interpretation": "Individual profile surprise",
            "why_flagged": why,
            "caveat": "This can still be real specialisation, pacing, illness, injury, or model uncertainty.",
            "confidence": confidence_for_case(row),
            "article_safe_sentence": "This case is a profile worth reviewing, not a conclusion about what happened.",
        }

    return {
        "primary_interpretation": "Low-priority review",
        "why_flagged": "No major V2 family dominates after context adjustment.",
        "caveat": "Keep only if useful as a borderline example.",
        "confidence": "low",
        "article_safe_sentence": "This case is useful mainly as a reminder that not every odd-looking profile is a strong signal.",
    }


if case_source.empty:
    case_explanation_table = pd.DataFrame()
else:
    explanation_rows = []
    for _, row in case_source.iterrows():
        explanation = explain_case(row)
        explanation_rows.append(
            {
                "case_id": row["case_id"],
                "distance": row["distance"],
                "gender": row["gender"],
                "cohort_size": row["cohort_valid_sbr_finishers"],
                "primary_interpretation": explanation["primary_interpretation"],
                "why_flagged": explanation["why_flagged"],
                "caveat": explanation["caveat"],
                "confidence": explanation["confidence"],
                "article_safe_sentence": explanation["article_safe_sentence"],
                "strongest_leg": row["best_leg"],
                "weakest_leg": row["worst_leg"],
                "swim_rel": round(row["swim_rel"], 3),
                "bike_rel": round(row["bike_rel"], 3),
                "run_rel": round(row["run_rel"], 3),
                "spread_rel": round(row["spread_rel"], 3),
                "max_model_abs_z": round(row["max_model_abs_z"], 2) if pd.notna(row["max_model_abs_z"]) else np.nan,
                "overall_delta_minutes": round(row["overall_delta_with_transitions_s"] / 60, 1) if pd.notna(row["overall_delta_with_transitions_s"]) else np.nan,
            }
        )
    case_explanation_table = pd.DataFrame(explanation_rows)

case_explanation_table

## 12. Validation <a id="validation"></a>

These checks make sure the V2 interpretation behaves as intended.

In [ ]:
IDENTIFYING_COLUMNS = {"Name", "Bib", "Country", "event_id", "source_file"}

article_tables = {
    "flag_taxonomy": flag_taxonomy,
    "cleaning_summary": cleaning_summary,
    "course_context_candidates": course_context_candidates,
    "individual_review_summary": individual_review_summary,
    "public_race_summary_v2": public_race_summary_v2.drop(columns=["event_id"], errors="ignore"),
    "before_after_hard_summary": before_after_hard_summary,
    "case_explanation_table": case_explanation_table,
}

for table_name, table in article_tables.items():
    leaked_columns = IDENTIFYING_COLUMNS.intersection(table.columns)
    assert not leaked_columns, f"{table_name} contains identifying columns: {sorted(leaked_columns)}"

expected_columns = [
    "event_context_flag",
    "event_context_affected_legs",
    "record_integrity_flag",
    "nominal_distance_hard_flag",
    "individual_hard_flag",
    "individual_profile_flag",
    "any_review_flag_v2",
    "reconciliation_flag",
    "duplicate_time_flag",
    "transition_review_flag",
    "transition_extreme_flag",
    "rank_time_mismatch_flag",
    "swim_event_context_flag",
    "bike_event_context_flag",
    "run_event_context_flag",
    "swim_individual_hard_flag",
    "bike_individual_hard_flag",
    "run_individual_hard_flag",
    "consistency_flag",
    "one_leg_spike_flag",
    "one_leg_collapse_flag",
    "multi_leg_gap_flag",
    "swim_model_robust_z",
    "bike_model_robust_z",
    "run_model_robust_z",
]

missing_columns = [col for col in expected_columns if col not in analysis_df.columns]
assert not missing_columns, f"Missing expected V2 analysis columns: {missing_columns}"

known_event_patterns = {
    "ironmanlouisville2018": "swim",
    "ironmansouthafrica2021": "swim",
    "ironmansouthafrica2023": "swim",
    "ironmannewzealand2012": "any",
}

known_event_checks = []
for pattern, expected_leg in known_event_patterns.items():
    matched = analysis_df.loc[analysis_df["event_id"].str.contains(pattern, case=False, regex=False)]
    if matched.empty:
        known_event_checks.append({"pattern": pattern, "present": False, "classified": np.nan})
        continue

    if expected_leg == "any":
        classified = bool(matched["event_context_flag"].any())
    else:
        classified = bool(matched[f"{expected_leg}_event_context_flag"].any())

    known_event_checks.append({"pattern": pattern, "present": True, "classified": classified})
    assert classified, f"Known modified event pattern was not classified as context: {pattern}"

known_event_checks = pd.DataFrame(known_event_checks)

if not case_explanation_table.empty:
    case_b = case_explanation_table.loc[case_explanation_table["case_id"].eq("Case B")]
    if not case_b.empty:
        assert case_b.iloc[0]["primary_interpretation"] == "Record integrity issue", "Case B should be record-integrity first."
    case_c = case_explanation_table.loc[case_explanation_table["case_id"].eq("Case C")]
    if not case_c.empty:
        assert case_c.iloc[0]["primary_interpretation"] == "Event/course context", "Case C should be event-context first."

assert before_after_hard_summary.loc[before_after_hard_summary["flag_family"].str.contains("Individual"), "rows"].iloc[0] <= before_after_hard_summary.loc[before_after_hard_summary["flag_family"].str.contains("Nominal"), "rows"].iloc[0]

print("Validation checks passed.")
display(known_event_checks)

## 13. Article summary <a id="summary"></a>

The improved article thesis is:

> Some anomalies are athletes, some are rows, and some are whole races.

The notebook now makes that distinction explicit. A 100% swim hard-rate is no longer presented as hundreds of individual red flags; it is treated as evidence that the swim was probably shortened, cancelled, or recorded under a different course context.